### Limpieza de los ficheros de censo de locales. 
## Los ficheros analizados son 06-200085-1-censo-locales.csv y 06-200085-6-censo-locales.csv. Se carga desde Kaggle y se limpia y normalizan los datos

In [45]:
import altair as alt

# Desactivar vegafusion si quedó activado accidentalmente
alt.data_transformers.enable("default", max_rows=None)

# Renderer compatible con JupyterLab / Anaconda
alt.renderers.enable("mimetype")

RendererRegistry.enable('mimetype')

In [2]:
!pip install reverse_geocoder

In [3]:
import pandas as pd
import reverse_geocoder as rg
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.decomposition import PCA
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [4]:
# Carga los datos raw disponibles en Kaggle. En la cuenta del alumno raquelahdo/ruido-datasets

file_path_061 = "06-200085-1-censo-locales.csv"
file_path_062 = "06-200085-6-censo-locales.csv"

df_061 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "raquelahdo/ruido-datasets",
  file_path_061,
    pandas_kwargs={"encoding": "latin-1", "sep": ";"}
)

df_062 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "raquelahdo/ruido-datasets",
  file_path_062,
    pandas_kwargs={"encoding": "latin-1", "sep": ";"}
)

C:\Users\Raquel\anaconda3\Lib\site-packages\kagglehub\pandas_datasets.py:92: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [5]:
df_061.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203092 entries, 0 to 203091
Data columns (total 46 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id_local                   203092 non-null  int64  
 1   id_distrito_local          203091 non-null  float64
 2   desc_distrito_local        203091 non-null  object 
 3   id_barrio_local            203091 non-null  float64
 4   desc_barrio_local          203091 non-null  object 
 5   cod_barrio_local           203091 non-null  float64
 6   id_seccion_censal_local    203091 non-null  float64
 7   desc_seccion_censal_local  203092 non-null  int64  
 8   coordenada_x_local         203092 non-null  float64
 9   coordenada_y_local         203092 non-null  float64
 10  id_tipo_acceso_local       203092 non-null  int64  
 11  desc_tipo_acceso_local     203092 non-null  object 
 12  id_situacion_local         203092 non-null  int64  
 13  desc_situacion_local       20

In [6]:
df_062.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6475 entries, 0 to 6474
Columns: 117 entries, id_terraza to fx_carga
dtypes: bool(7), float64(39), int64(33), object(38)
memory usage: 5.5+ MB


In [7]:
df_061.describe()

,id_local,id_distrito_local,id_barrio_local,cod_barrio_local,id_seccion_censal_local,desc_seccion_censal_local,coordenada_x_local,coordenada_y_local,id_tipo_acceso_local,id_situacion_local,id_vial_edificio,id_ndp_edificio,id_clase_ndp_edificio,num_edificio,secuencial_local_PC,id_vial_acceso,id_ndp_acceso,id_clase_ndp_acceso,num_acceso,coordenada_x_agrupacion,coordenada_y_agrupacion,id_agrupacion,id_tipo_agrup,cod_postal
count,2.030920e+05,203091.000000,203091.000000,203091.000000,203091.000000,203092.000000,203092.000000,2.030920e+05,203092.000000,203092.000000,2.030580e+05,2.030920e+05,203092.0,203058.000000,203092.000000,2.030630e+05,2.030910e+05,203091.0,203063.000000,14603.000000,1.460300e+04,1.460300e+04,14603.000000,203058.000000
mean,2.693311e+08,9.138263,917.307837,3.481523,9209.003058,70.739566,332198.716728,3.364525e+06,1.585749,2.323779,1.133050e+06,1.401738e+07,1.0,42.097952,16.937176,1.133037e+06,1.454293e+07,1.0,41.851411,393841.054365,3.983451e+06,9.900024e+07,7.101623,28023.650253
std,4.458614e+07,5.766528,576.549849,1.877988,5769.436585,47.441079,190844.608059,1.932647e+06,1.558995,2.214684,4.749373e+06,6.170209e+06,0.0,87.449071,56.979441,4.755654e+06,6.569380e+06,0.0,87.388405,138304.034616,1.398361e+06,1.748444e+02,5.508651,14.715349
min,1.000000e+07,1.000000,101.000000,1.000000,1001.000000,0.000000,0.000000,0.000000e+00,0.000000,1.000000,1.270000e+02,3.012800e+04,1.0,1.000000,0.000000,1.270000e+02,1.100000e+07,1.0,1.000000,0.000000,0.000000e+00,9.900000e+07,1.000000,28001.000000
25%,2.705444e+08,4.000000,405.000000,2.000000,4110.000000,32.000000,432053.660000,4.465063e+06,1.000000,1.000000,1.538000e+05,1.102209e+07,1.0,8.000000,0.000000,1.536000e+05,1.102292e+07,1.0,8.000000,438973.610000,4.469252e+06,9.900010e+07,1.000000,28011.000000
50%,2.800285e+08,8.000000,807.000000,3.000000,8171.000000,66.000000,440242.615000,4.472695e+06,1.000000,1.000000,3.825000e+05,1.106822e+07,1.0,20.000000,10.000000,3.785000e+05,1.107184e+07,1.0,20.000000,441296.580000,4.473187e+06,9.900021e+07,4.000000,28022.000000
75%,2.850109e+08,14.000000,1402.000000,5.000000,14013.000000,105.000000,443100.580000,4.476350e+06,3.000000,4.000000,6.102000e+05,1.113281e+07,1.0,47.000000,20.000000,6.100000e+05,2.000425e+07,1.0,47.000000,444449.570000,4.477306e+06,9.900033e+07,12.000000,28036.000000
max,3.000220e+08,21.000000,2105.000000,9.000000,21124.000000,221.000000,454736.523724,4.495497e+06,12.000000,9.000000,3.100611e+07,9.190070e+07,1.0,5090.000000,9010.000000,3.100611e+07,3.107544e+07,1.0,5090.000000,452667.560000,4.488549e+06,9.900078e+07,17.000000,28055.000000


In [8]:
df_062.describe()

,id_terraza,id_local,id_distrito_local,id_barrio_local,id_ndp_edificio,id_clase_ndp_edificio,id_vial_edificio,num_edificio,Cod_Postal,coordenada_x_local,coordenada_y_local,id_tipo_acceso_local,id_situacion_local,secuencial_local_PC,id_planta_agrupado,coordenada_x_agrupacion,coordenada_y_agrupacion,id_periodo_terraza,id_situacion_terraza,Superficie_ES,Superficie_RA,id_ndp_terraza,id_clase_ndp_terraza,id_vial,num_terraza,mesas_aux_es,mesas_aux_ra,mesas_es,mesas_ra,sillas_es,sillas_ra,mesas_altas_estacional,mesas_altas_anual,ambito_ordenacion_cod,denominacion_cod,tipo_suelo_cod,separadores_moviles_menor_estacional,separadores_moviles_mayor_estacional,auxiliares_informacion_es,elem_jard_menor_estacional,elem_jard_mayor_estacional,sombrillas_es,separadores_moviles_menor_anual,separadores_moviles_mayor_anual,auxiliares_informacion_ra,elem_jard_menor_anual,elem_jard_mayor_anual,sombrillas_ra,toldos_pavimento_es,sombrillas_pavimento_es,elem_separacion_menor_estacional,elem_separacion_mayor_estacional,elem_aux_apoyo_es,toldos_pavimento_ra,sombrillas_pavimento_ra,elem_separacion_menor_anual,elem_separacion_mayor_anual,elem_aux_apoyo_ra,estufas_gas_es,estufas_electricas_es,aparatos_clima_es,nebulizadores_es,iluminacion_es,estufas_gas_ra,estufas_electricas_ra,aparatos_clima_ra,nebulizadores_ra,iluminacion_ra,sonometro_estacional,sonometro_anual,placas_fotovoltaicas_estacional,placas_fotovoltaicas_anual
count,6475.000000,6.475000e+03,6475.000000,6475.000000,6.475000e+03,6475.0,6.475000e+03,6475.000000,6475.000000,6475.000000,6.475000e+03,6475.000000,6475.000000,6475.000000,6466.000000,107.000000,1.070000e+02,6475.000000,6475.000000,6475.000000,5445.000000,6.475000e+03,6475.0,6.473000e+03,6473.000000,6475.000000,6339.000000,6475.000000,6339.000000,6475.000000,6339.000000,3364.000000,2961.000000,46.000000,45.000000,6475.000000,6475.000000,3364.000000,6475.000000,6475.000000,3364.000000,6475.000000,6206.000000,2961.000000,6339.00000,6206.000000,2961.000000,6339.000000,6475.000000,6475.000000,6475.000000,3364.000000,6475.000000,6339.000000,6339.000000,6206.000000,2961.000000,6339.000000,6475.000000,6475.000000,6475.000000,6475.000000,6475.000000,6339.000000,6339.000000,6339.000000,5445.000000,5445.000000,3364.000000,2961.000000,3364.000000,2961.000000
mean,8041.528340,2.363660e+08,8.441390,847.702394,1.426967e+07,1.0,1.862688e+06,43.802625,28023.231351,434515.342004,4.400351e+06,1.001390,1.103320,21.888803,1.038200,388520.655888,3.931848e+06,1.159073,1.033514,30.365586,29.157293,1.525424e+07,1.0,1.817828e+06,44.764252,0.042934,0.037388,9.862085,8.113898,31.656525,25.727244,0.133769,0.146572,5.195652,3.977778,1.105637,0.860386,0.052616,0.022239,0.633822,0.033294,2.017915,0.829198,0.059777,0.02177,0.594586,0.037487,1.624389,0.056680,0.037529,0.183938,0.011891,0.006486,0.055687,0.034864,0.094908,0.013509,0.006152,0.159073,0.448958,0.021313,0.027799,0.084479,0.176211,0.469475,0.020666,0.032323,0.099174,0.005648,0.006417,0.001189,0.001351
std,7336.186775,8.004736e+07,5.815043,581.513582,6.418922e+06,0.0,6.604868e+06,64.821560,15.427676,56683.989673,5.731577e+05,0.388073,0.668118,20.746793,2.223949,145201.779137,1.469079e+06,0.365773,0.483226,31.541831,30.597286,7.089696e+06,0.0,6.505426e+06,66.595418,0.283972,0.262323,7.695830,7.639848,25.910560,25.186835,0.779190,0.812983,1.469990,2.061430,0.307396,3.039636,0.687531,0.182996,2.825116,0.718117,2.599872,3.232065,0.732549,0.18159,2.749082,0.765126,2.405115,0.356804,0.377765,1.430175,0.283097,0.108154,0.352009,0.364765,0.987521,0.301718,0.107145,0.914997,1.915011,0.338330,0.164410,0.278126,0.984590,1.928627,0.311830,0.176873,0.298922,0.074952,0.079861,0.034467,0.036736
min,8.000000,1.000000e+07,1.000000,101.000000,1.100000e+07,1.0,1.270000e+02,1.000000,28001.000000,0.000000,0.000000e+00,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,1.000000,0.750000,0.750000,0.000000e+00,1.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,2.00

In [9]:
missing_values_061 = df_061.isnull().sum()
display(missing_values_061)

id_local                          0
id_distrito_local                 1
desc_distrito_local               1
id_barrio_local                   1
desc_barrio_local                 1
cod_barrio_local                  1
id_seccion_censal_local           1
desc_seccion_censal_local         0
coordenada_x_local                0
coordenada_y_local                0
id_tipo_acceso_local              0
desc_tipo_acceso_local            0
id_situacion_local                0
desc_situacion_local              0
id_vial_edificio                 34
clase_vial_edificio              34
desc_vial_edificio               34
id_ndp_edificio                   0
id_clase_ndp_edificio             0
nom_edificio                     34
num_edificio                     34
cal_edificio                     34
secuencial_local_PC               0
id_vial_acceso                   29
clase_vial_acceso                29
desc_vial_acceso                 29
id_ndp_acceso                     1
id_clase_ndp_acceso         

In [10]:
missing_values_062 = df_062.isnull().sum()
display(missing_values_062)

id_terraza                                 0
id_local                                   0
id_distrito_local                          0
desc_distrito_local                        0
id_barrio_local                            0
desc_barrio_local                          0
id_ndp_edificio                            0
id_clase_ndp_edificio                      0
id_vial_edificio                           0
clase_vial_edificio                        0
desc_vial_edificio                         0
nom_edificio                               0
num_edificio                               0
cal_edificio                               0
ref_catastral                            540
Cod_Postal                                 0
coordenada_x_local                         0
coordenada_y_local                         0
id_tipo_acceso_local                       0
desc_tipo_acceso_local                     0
id_situacion_local                         0
desc_situacion_local                       0
secuencial

In [11]:
df_061.head()

,id_local,id_distrito_local,desc_distrito_local,id_barrio_local,desc_barrio_local,cod_barrio_local,id_seccion_censal_local,desc_seccion_censal_local,coordenada_x_local,coordenada_y_local,id_tipo_acceso_local,desc_tipo_acceso_local,id_situacion_local,desc_situacion_local,id_vial_edificio,clase_vial_edificio,desc_vial_edificio,id_ndp_edificio,id_clase_ndp_edificio,nom_edificio,num_edificio,cal_edificio,secuencial_local_PC,id_vial_acceso,clase_vial_acceso,desc_vial_acceso,id_ndp_acceso,id_clase_ndp_acceso,nom_acceso,num_acceso,cal_acceso,coordenada_x_agrupacion,coordenada_y_agrupacion,id_agrupacion,nombre_agrupacion,id_tipo_agrup,desc_tipo_agrup,id_planta_agrupado,id_local_agrupado,rotulo,cod_postal,hora_apertura1,hora_apertura2,hora_cierre1,hora_cierre2,fx_carga
0,10000003,1.0,CENTRO,104.0,JUSTICIA,4.0,1088.0,88,440554.59,4475338.53,1,Puerta Calle,1,Abierto,92900.0,CALLE,BARCELO ...,11005235,1,NUM,5.0,,10,92900.0,CALLE,BARCELO ...,11005235.0,1.0,NUM,5.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,03,VITACA,28004.0,NaN,NaN,NaN,NaN,05/03/2026
1,10000004,1.0,CENTRO,105.0,UNIVERSIDAD,5.0,1106.0,106,439945.60,4475591.53,1,Puerta Calle,1,Abierto,4700.0,CALLE,ACUERDO ...,11006452,1,NUM,31.0,,10,4700.0,CALLE,ACUERDO ...,11006452.0,1.0,NUM,31.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,02,ZAATAR & CO,28015.0,10:00,00:00,02:00,00:00,05/03/2026
2,10000013,1.0,CENTRO,102.0,EMBAJADORES,2.0,1058.0,58,441199.58,4473326.52,1,Puerta Calle,1,Abierto,264800.0,PLAZA,EMPERADOR CARLOS V ...,11003119,1,NUM,8.0,,20,264800.0,PLAZA,EMPERADOR CARLOS V ...,11003119.0,1.0,NUM,8.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,07,HOTEL MEDIODIA,28012.0,NaN,NaN,NaN,NaN,05/03/2026
3,10000044,1.0,CENTRO,101.0,PALACIO,1.0,1014.0,14,439722.59,4473550.53,1,Puerta Calle,4,Cerrado,369700.0,CALLE,HUMILLADERO ...,11000878,1,NUM,16.0,,10,369700.0,CALLE,HUMILLADERO ...,11000878.0,1.0,NUM,16.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,,V.M. VINOMANIA,28005.0,NaN,NaN,NaN,NaN,05/03/2026
4,10000052,1.0,CENTRO,104.0,JUSTICIA,4.0,1082.0,82,440875.59,4475142.52,1,Puerta Calle,1,Abierto,678900.0,TRAVESIA,SAN MATEO ...,11005006,1,NUM,17.0,,10,576200.0,CALLE,PELAYO ...,11004968.0,1.0,NUM,57.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,IZ,MUNE,28004.0,NaN,NaN,NaN,NaN,05/03/2026


In [12]:
df_062.head()

,id_terraza,id_local,id_distrito_local,desc_distrito_local,id_barrio_local,desc_barrio_local,id_ndp_edificio,id_clase_ndp_edificio,id_vial_edificio,clase_vial_edificio,desc_vial_edificio,nom_edificio,num_edificio,cal_edificio,ref_catastral,Cod_Postal,coordenada_x_local,coordenada_y_local,id_tipo_acceso_local,desc_tipo_acceso_local,id_situacion_local,desc_situacion_local,secuencial_local_PC,Escalera,id_planta_agrupado,id_local_agrupado,coordenada_x_agrupacion,coordenada_y_agrupacion,rotulo,id_periodo_terraza,desc_periodo_terraza,id_situacion_terraza,desc_situacion_terraza,Superficie_ES,Superficie_RA,Fecha_confir_ult_decreto_resol,id_ndp_terraza,id_clase_ndp_terraza,id_vial,desc_clase,desc_nombre,nom_terraza,num_terraza,cal_terraza,desc_ubicacion_terraza,hora_ini_LJ_es,hora_fin_LJ_es,hora_ini_LJ_ra,hora_fin_LJ_ra,hora_ini_VS_es,hora_fin_VS_es,hora_ini_VS_ra,hora_fin_VS_ra,mesas_aux_es,mesas_aux_ra,mesas_es,mesas_ra,sillas_es,sillas_ra,mesas_altas_estacional,mesas_altas_anual,anclaje,apilamiento,ambito_ordenacion_cod,ambito_ordenacion_desc,denominacion_cod,denominacion_desc,fecha_acuerdo_delimitacion,tipo_suelo_cod,tipo_suelo_desc,separadores_moviles_menor_estacional,separadores_moviles_mayor_estacional,auxiliares_informacion_es,elem_jard_menor_estacional,elem_jard_mayor_estacional,sombrillas_es,otros_es_mobiliario,separadores_moviles_menor_anual,separadores_moviles_mayor_anual,auxiliares_informacion_ra,elem_jard_menor_anual,elem_jard_mayor_anual,sombrillas_ra,otros_ra_mobiliario,construccion_ligera_fachada_es,construccion_ligera_bordillo_es,tarima_pavimento_es,toldos_pavimento_es,sombrillas_pavimento_es,elem_separacion_menor_estacional,elem_separacion_mayor_estacional,elem_aux_apoyo_es,construccion_ligera_fachada_ra,construccion_ligera_bordillo_ra,tarima_pavimento_ra,toldos_pavimento_ra,sombrillas_pavimento_ra,elem_separacion_menor_anual,elem_separacion_mayor_anual,elem_aux_apoyo_ra,estufas_gas_es,estufas_electricas_es,aparatos_clima_es,nebulizadores_es,iluminacion_es,otros_es_elem_ind,estufas_gas_ra,estufas_electricas_ra,aparatos_clima_ra,nebulizadores_ra,iluminacion_ra,otros_ra_elem_ind,sonometro_estacional,sonometro_anual,placas_fotovoltaicas_estacional,placas_fotovoltaicas_anual,fx_carga
0,37,270184111,18,VILLA DE VALLECAS,1801,CASCO H.VALLECAS,11129905,1,2872,CALLE,SIERRA GADOR ...,NUM,41,,7303206VK4770C0006SP,28031,447209.53,4470130.43,1,Puerta Calle,1,Abierto,10,NaN,1.0,01,NaN,NaN,LOS MANCHEGOS,1,Anual,1,Abierta,16.70,16.70,2014-04-07 09:27:35.943,11129905,1,2872.0,CALLE,SIERRA GADOR ...,NUM,41.0,,Plaza peatonal,08:00:00,01:00:00,08:00:00,00:00:00,08:00:00,01:30:00,08:00:00,00:00:00,0,0.0,6,6.0,19,19.0,0.0,0.0,False,False,NaN,NaN,NaN,NaN,NaN,1,Suelo publico,0,0.0,0,0,0.0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,False,False,True,False,0,0,0,0.0,0,False,True,False,0.0,0.0,0.0,0.0,0.0,0,0,0,0,1,False,0.0,0.0,0.0,0.0,1.0,False,0.0,0.0,0.0,0.0,05/03/2026
1,42,40002970,4,SALAMANCA,403,FUENTE DEL BERRO,11016907,1,18900,CALLE,ALCALA ...,NUM,200,A,3859807VK4735H0030,28028,443685.58,4475723.49,1,Puerta Calle,1,Abierto,20,,1.0,,NaN,NaN,VIPS,1,Anual,1,Abierta,14.89,14.89,2014-04-07 10:02:16.493,11016907,1,18900.0,CALLE,ALCALA ...,NUM,200.0,A,Acera,08:00:00,01:00:00,08:00:00,00:00:00,08:00:00,01:30:00,08:00:00,00:00:00,0,0.0,6,6.0,18,18.0,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,1,Suelo publico,0,NaN,0,0,NaN,2,False,0.0,NaN,0.0,0.0,NaN,2.0,False,False,False,False,0,0,0,NaN,0,False,False,False,0.0,0.0,0.0,NaN,0.0,0,0,0,0,0,False,0.0,0.0,0.0,0.0,0.0,False,NaN,NaN,NaN,NaN,05/03/2026
2,44,40001459,4,SALAMANCA,404,GUINDALERA,11018130,1,42500,AVENIDA,AMERICA ...,NUM,32,,NaN,28028,442926.58,4476681.50,1,Puerta Calle,1,Abierto,10,,1.0,,NaN,NaN,HOTEL ABBA MADRID,2,Estacional,1,Abierta,19.44,NaN,2014-04-07 10:31:51.320,11018130,1,42500.0,AVENIDA,AMERICA ...,NUM,32.0,,Acera,10:00:00,01:00:00,NaN,NaN,10:00:00,02:30:00,NaN,NaN,0,0.0,6,0.0,24,0.0,NaN,NaN,False,False,NaN,NaN,NaN,NaN,NaN,1,Suelo publico,0,NaN,0,0,NaN,0,False,0.0,NaN,0.0,0.0,NaN,0.0,NaN,False,F

Renombrado estandarizado de columnas: Nombres en minusculas, sin espacios, sin acentos, consistentes con Python.
Convertir tipos numéricos correctamente (a float)
Creación de columna Fecha.
Ordenar dataset por fecha.

Dificultades del dataset original:
A) LONGITUD y LATITUD vienen como texto se pasan a número float. 
B) FECHA ALTA está en formato DD/MM/AAAA, no datetime.
D) X, Y, ALTITUD deben ser numéricos.
E) UBICACIÓN, DISTRITO, BARRIO contienen texto largo → mantener como string.

In [13]:
#creamos una columna fecha para trabajar
#renombramos las columnas es formato snake_case, sin acentos, consistente con los datos trabajados.
import unicodedata
import re

def limpiar_columnas(df):
    nuevas = []
    for col in df.columns:
        # 1. Convertir a str
        col = str(col)

        # 2. Normalizar unicode (elimina acentos)
        col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')

        # 3. Minúsculas
        col = col.lower()

        # 4. Reemplazar cualquier separador por _
        col = re.sub(r'[^a-z0-9]+', '_', col)

        # 5. Eliminar _ inicial o final
        col = col.strip('_')

        nuevas.append(col)

    df.columns = nuevas
    return df

# Aplicación a df_061 y df_062:
df_061 = limpiar_columnas(df_061)
df_062 = limpiar_columnas(df_062)

## LIMPIEZA DE FICHERO 061
#Se convierten coordenadas UTM a numéricas
df_061["coordenada_x_local"] = pd.to_numeric(df_061["coordenada_x_local"], errors="coerce")
df_061["coordenada_y_local"] = pd.to_numeric(df_061["coordenada_y_local"], errors="coerce")
df_061["coordenada_x_agrupacion"] = pd.to_numeric(df_061["coordenada_x_agrupacion"], errors="coerce")
df_061["coordenada_y_agrupacion"] = pd.to_numeric(df_061["coordenada_y_agrupacion"], errors="coerce")

#Convertir códigos numéricos a enteros o floats
# Lista limpia de columnas que deben ser numéricas
cols_num = [
    "id_local","id_distrito_local","id_barrio_local","cod_barrio_local",
    "id_seccion_censal_local","id_tipo_acceso_local","id_situacion_local",
    "id_vial_edificio","id_ndp_edificio","num_edificio",
    "id_vial_acceso","id_ndp_acceso","num_acceso"
]

for col in cols_num:
    if col in df_061.columns:
        df_061[col] = (
            df_061[col]
            .astype(str)
            .str.strip()                 # elimina espacios
            .replace("", np.nan)         # strings vacíos → NaN
            .replace("nan", np.nan)      # strings "nan" → NaN
        )
        df_061[col] = pd.to_numeric(df_061[col], errors="coerce")


#Se Normaliza el texto en las columnas
cols_text = [
    "desc_distrito_local","desc_barrio_local","desc_seccion_censal_local",
    "desc_tipo_acceso_local","desc_situacion_local","clase_vial_edificio",
    "desc_vial_edificio","nom_edificio","cal_edificio",
    "clase_vial_acceso","desc_vial_acceso","nom_acceso","cal_acceso",
    "rotulo"
]

for col in cols_text:
    df_061[col] = df_061[col].astype(str).str.strip()

#Convertir fecha fx_carga
df_061["fx_carga"] = pd.to_datetime(df_061["fx_carga"], format="%d/%m/%Y", errors="coerce")




In [14]:
df_061.head()

,id_local,id_distrito_local,desc_distrito_local,id_barrio_local,desc_barrio_local,cod_barrio_local,id_seccion_censal_local,desc_seccion_censal_local,coordenada_x_local,coordenada_y_local,id_tipo_acceso_local,desc_tipo_acceso_local,id_situacion_local,desc_situacion_local,id_vial_edificio,clase_vial_edificio,desc_vial_edificio,id_ndp_edificio,id_clase_ndp_edificio,nom_edificio,num_edificio,cal_edificio,secuencial_local_pc,id_vial_acceso,clase_vial_acceso,desc_vial_acceso,id_ndp_acceso,id_clase_ndp_acceso,nom_acceso,num_acceso,cal_acceso,coordenada_x_agrupacion,coordenada_y_agrupacion,id_agrupacion,nombre_agrupacion,id_tipo_agrup,desc_tipo_agrup,id_planta_agrupado,id_local_agrupado,rotulo,cod_postal,hora_apertura1,hora_apertura2,hora_cierre1,hora_cierre2,fx_carga
0,10000003,1.0,CENTRO,104.0,JUSTICIA,4.0,1088.0,88,440554.59,4475338.53,1,Puerta Calle,1,Abierto,92900.0,CALLE,BARCELO,11005235,1,NUM,5.0,,10,92900.0,CALLE,BARCELO,11005235.0,1.0,NUM,5.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,03,VITACA,28004.0,NaN,NaN,NaN,NaN,2026-03-05
1,10000004,1.0,CENTRO,105.0,UNIVERSIDAD,5.0,1106.0,106,439945.60,4475591.53,1,Puerta Calle,1,Abierto,4700.0,CALLE,ACUERDO,11006452,1,NUM,31.0,,10,4700.0,CALLE,ACUERDO,11006452.0,1.0,NUM,31.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,02,ZAATAR & CO,28015.0,10:00,00:00,02:00,00:00,2026-03-05
2,10000013,1.0,CENTRO,102.0,EMBAJADORES,2.0,1058.0,58,441199.58,4473326.52,1,Puerta Calle,1,Abierto,264800.0,PLAZA,EMPERADOR CARLOS V,11003119,1,NUM,8.0,,20,264800.0,PLAZA,EMPERADOR CARLOS V,11003119.0,1.0,NUM,8.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,07,HOTEL MEDIODIA,28012.0,NaN,NaN,NaN,NaN,2026-03-05
3,10000044,1.0,CENTRO,101.0,PALACIO,1.0,1014.0,14,439722.59,4473550.53,1,Puerta Calle,4,Cerrado,369700.0,CALLE,HUMILLADERO,11000878,1,NUM,16.0,,10,369700.0,CALLE,HUMILLADERO,11000878.0,1.0,NUM,16.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,,V.M. VINOMANIA,28005.0,NaN,NaN,NaN,NaN,2026-03-05
4,10000052,1.0,CENTRO,104.0,JUSTICIA,4.0,1082.0,82,440875.59,4475142.52,1,Puerta Calle,1,Abierto,678900.0,TRAVESIA,SAN MATEO,11005006,1,NUM,17.0,,10,576200.0,CALLE,PELAYO,11004968.0,1.0,NUM,57.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,IZ,MUNE,28004.0,NaN,NaN,NaN,NaN,2026-03-05


In [15]:
## LIMPIEZA FICHERO DF_062 (TERRAZAS)
#Renombrar columnas YA REALIZADO con df_061

#Conversión de coordenadas
for col in ["coordenada_x_local","coordenada_y_local",
            "coordenada_x_agrupacion","coordenada_y_agrupacion"]:
    df_062[col] = pd.to_numeric(df_062[col], errors="coerce")

#Variables de superficie (float)
df_062["superficie_es"] = pd.to_numeric(df_062["superficie_es"], errors="coerce")
df_062["superficie_ra"] = pd.to_numeric(df_062["superficie_ra"], errors="coerce")

 #Conversión de campos booleanos
cols_bool = [c for c in df_062.columns if df_062[c].dtype == object and df_062[c].isin(["True","False","true","false","1","0"]).any()]

for col in cols_bool:
    df_062[col] = df_062[col].astype(str).str.lower().map({"true":True,"false":False,"1":True,"0":False})

#Tiempos de apertura/cierre
cols_time = [c for c in df_062.columns if "hora" in c.lower()]

for col in cols_time:
    df_062[col] = pd.to_datetime(df_062[col], format="%H:%M:%S", errors="coerce").dt.time
    
#Fechas
for col in ["fx_carga","fecha_confir_ult_decreto_resol"]:
    df_062[col] = pd.to_datetime(df_062[col], errors="coerce")



In [16]:
df_062.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6475 entries, 0 to 6474
Columns: 117 entries, id_terraza to fx_carga
dtypes: bool(7), datetime64[ns](2), float64(39), int64(33), object(36)
memory usage: 5.5+ MB


In [17]:
##UNIMOS DF_061 Y DF_062 POR EL CAMPO ID_LOCAL

df_locales_terrazas = df_061.merge(
    df_062,
    on="id_local",
    how="left",
    suffixes=("_local","_terraza")
)


In [18]:
df_locales_terrazas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203241 entries, 0 to 203240
Columns: 162 entries, id_local to fx_carga_terraza
dtypes: datetime64[ns](3), float64(88), int64(6), object(65)
memory usage: 251.2+ MB


In [19]:
df_locales_terrazas.head()

,id_local,id_distrito_local_local,desc_distrito_local_local,id_barrio_local_local,desc_barrio_local_local,cod_barrio_local,id_seccion_censal_local,desc_seccion_censal_local,coordenada_x_local_local,coordenada_y_local_local,id_tipo_acceso_local_local,desc_tipo_acceso_local_local,id_situacion_local_local,desc_situacion_local_local,id_vial_edificio_local,clase_vial_edificio_local,desc_vial_edificio_local,id_ndp_edificio_local,id_clase_ndp_edificio_local,nom_edificio_local,num_edificio_local,cal_edificio_local,secuencial_local_pc_local,id_vial_acceso,clase_vial_acceso,desc_vial_acceso,id_ndp_acceso,id_clase_ndp_acceso,nom_acceso,num_acceso,cal_acceso,coordenada_x_agrupacion_local,coordenada_y_agrupacion_local,id_agrupacion,nombre_agrupacion,id_tipo_agrup,desc_tipo_agrup,id_planta_agrupado_local,id_local_agrupado_local,rotulo_local,cod_postal_local,hora_apertura1,hora_apertura2,hora_cierre1,hora_cierre2,fx_carga_local,id_terraza,id_distrito_local_terraza,desc_distrito_local_terraza,id_barrio_local_terraza,desc_barrio_local_terraza,id_ndp_edificio_terraza,id_clase_ndp_edificio_terraza,id_vial_edificio_terraza,clase_vial_edificio_terraza,desc_vial_edificio_terraza,nom_edificio_terraza,num_edificio_terraza,cal_edificio_terraza,ref_catastral,cod_postal_terraza,coordenada_x_local_terraza,coordenada_y_local_terraza,id_tipo_acceso_local_terraza,desc_tipo_acceso_local_terraza,id_situacion_local_terraza,desc_situacion_local_terraza,secuencial_local_pc_terraza,escalera,id_planta_agrupado_terraza,id_local_agrupado_terraza,coordenada_x_agrupacion_terraza,coordenada_y_agrupacion_terraza,rotulo_terraza,id_periodo_terraza,desc_periodo_terraza,id_situacion_terraza,desc_situacion_terraza,superficie_es,superficie_ra,fecha_confir_ult_decreto_resol,id_ndp_terraza,id_clase_ndp_terraza,id_vial,desc_clase,desc_nombre,nom_terraza,num_terraza,cal_terraza,desc_ubicacion_terraza,hora_ini_lj_es,hora_fin_lj_es,hora_ini_lj_ra,hora_fin_lj_ra,hora_ini_vs_es,hora_fin_vs_es,hora_ini_vs_ra,hora_fin_vs_ra,mesas_aux_es,mesas_aux_ra,mesas_es,mesas_ra,sillas_es,sillas_ra,mesas_altas_estacional,mesas_altas_anual,anclaje,apilamiento,ambito_ordenacion_cod,ambito_ordenacion_desc,denominacion_cod,denominacion_desc,fecha_acuerdo_delimitacion,tipo_suelo_cod,tipo_suelo_desc,separadores_moviles_menor_estacional,separadores_moviles_mayor_estacional,auxiliares_informacion_es,elem_jard_menor_estacional,elem_jard_mayor_estacional,sombrillas_es,otros_es_mobiliario,separadores_moviles_menor_anual,separadores_moviles_mayor_anual,auxiliares_informacion_ra,elem_jard_menor_anual,elem_jard_mayor_anual,sombrillas_ra,otros_ra_mobiliario,construccion_ligera_fachada_es,construccion_ligera_bordillo_es,tarima_pavimento_es,toldos_pavimento_es,sombrillas_pavimento_es,elem_separacion_menor_estacional,elem_separacion_mayor_estacional,elem_aux_apoyo_es,construccion_ligera_fachada_ra,construccion_ligera_bordillo_ra,tarima_pavimento_ra,toldos_pavimento_ra,sombrillas_pavimento_ra,elem_separacion_menor_anual,elem_separacion_mayor_anual,elem_aux_apoyo_ra,estufas_gas_es,estufas_electricas_es,aparatos_clima_es,nebulizadores_es,iluminacion_es,otros_es_elem_ind,estufas_gas_ra,estufas_electricas_ra,aparatos_clima_ra,nebulizadores_ra,iluminacion_ra,otros_ra_elem_ind,sonometro_estacional,sonometro_anual,placas_fotovoltaicas_estacional,placas_fotovoltaicas_anual,fx_carga_terraza
0,10000003,1.0,CENTRO,104.0,JUSTICIA,4.0,1088.0,88,440554.59,4475338.53,1,Puerta Calle,1,Abierto,92900.0,CALLE,BARCELO,11005235,1,NUM,5.0,,10,92900.0,CALLE,BARCELO,11005235.0,1.0,NUM,5.0,,NaN,NaN,NaN,NaN,NaN,NaN,1,03,VITACA,28004.0,NaN,NaN,NaN,NaN,2026-03-05,2038.0,1.0,CENTRO,104.0,JUSTICIA,11005235.0,1.0,92900.0,CALLE,BARCELO ...,NUM,5.0,,0655407VK4705F8011NN,28004.0,440554.59,4475338.53,1.0,Puerta Calle,1.0,Abierto,10.0,NaN,1.0,NaN,NaN,NaN,VITACA,1.0,Anual,1.0,Abierta,6.48,6.48,2015-04-07 13:18:33.267,11005235.0,1.0,92900.0,CALLE,BARCELO ...,NUM,5.0,,Acera,10:00:00,00:30:00,10:00:00,23:00:00,10:00:00,01:30:00,10:00:00,23:00:00,0.0,0.0,2

NECESIDAD DE HABILITAR LA GESTIÓN MASIVA DE DATOS
The number of rows in your dataset is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

In [20]:
pip install "vegafusion[embed]>=1.5.0"

Note: you may need to restart the kernel to use updated packages.


In [21]:
 pip install "vl-convert-python>=1.6.0"

Note: you may need to restart the kernel to use updated packages.


In [22]:
##Función para convertir horas a decimal

from datetime import datetime
import numpy as np

def hora_a_decimal(h):
    if pd.isna(h):
        return np.nan
    if isinstance(h, str):
        h = h.strip()
        if h == "":
            return np.nan
        # Intentar HH:MM:SS
        try:
            t = datetime.strptime(h, "%H:%M:%S").time()
            return t.hour + t.minute/60 + t.second/3600
        except:
            pass
        # Intentar HH:MM
        try:
            t = datetime.strptime(h, "%H:%M").time()
            return t.hour + t.minute/60
        except:
            return np.nan

    # Si realmente es datetime.time
    if hasattr(h, "hour"):
        return h.hour + h.minute/60 + h.second/3600
    return np.nan

In [23]:
[col for col in df_locales_terrazas.columns if "rotulo" in col.lower()]

['rotulo_local', 'rotulo_terraza']

In [24]:
# Crear columna rotulo unificada (terraza tiene prioridad)
df_locales_terrazas["rotulo"] = df_locales_terrazas["rotulo_terraza"].fillna(
    df_locales_terrazas["rotulo_local"]
)

#Asegurar que es una columna STRING válida para Altair
df_locales_terrazas["rotulo"] = (
    df_locales_terrazas["rotulo"]
    .astype(str)
    .replace("nan", "")
    .replace("None", "")
    .str.strip()
)

#Se crea columna numérica hora_cierre_num
df_locales_terrazas["hora_cierre_num"] = df_locales_terrazas["hora_cierre1"].apply(hora_a_decimal)


In [25]:
df_locales_terrazas.columns.tolist()

['id_local',
 'id_distrito_local_local',
 'desc_distrito_local_local',
 'id_barrio_local_local',
 'desc_barrio_local_local',
 'cod_barrio_local',
 'id_seccion_censal_local',
 'desc_seccion_censal_local',
 'coordenada_x_local_local',
 'coordenada_y_local_local',
 'id_tipo_acceso_local_local',
 'desc_tipo_acceso_local_local',
 'id_situacion_local_local',
 'desc_situacion_local_local',
 'id_vial_edificio_local',
 'clase_vial_edificio_local',
 'desc_vial_edificio_local',
 'id_ndp_edificio_local',
 'id_clase_ndp_edificio_local',
 'nom_edificio_local',
 'num_edificio_local',
 'cal_edificio_local',
 'secuencial_local_pc_local',
 'id_vial_acceso',
 'clase_vial_acceso',
 'desc_vial_acceso',
 'id_ndp_acceso',
 'id_clase_ndp_acceso',
 'nom_acceso',
 'num_acceso',
 'cal_acceso',
 'coordenada_x_agrupacion_local',
 'coordenada_y_agrupacion_local',
 'id_agrupacion',
 'nombre_agrupacion',
 'id_tipo_agrup',
 'desc_tipo_agrup',
 'id_planta_agrupado_local',
 'id_local_agrupado_local',
 'rotulo_local',
 '

In [28]:
#Creamos una función para extraer un dataset reducido con columnas realionadas con el estudio de ruido, nivel socioeconómico

def crear_df_locales_master(df):
    """
    Crea un dataset optimizado desde df_locales_terrazas
    para análisis de ruido, ubicación, renta y calidad de vida.
    """

    # -----------------------
    # 1. Definir columnas útiles
    # -----------------------
    columnas = [

        # Identificación
        "id_local",
        "rotulo_local",
        "id_terraza",
        "rotulo_terraza",

        # Ubicación administrativa
        "id_distrito_local_local",
        "desc_distrito_local_local",
        "id_barrio_local_local",
        "desc_barrio_local_local",
        "cod_barrio_local",
        "id_seccion_censal_local",

        # Coordenadas
        "coordenada_x_local_local",
        "coordenada_y_local_local",

        # Información del local
        "desc_tipo_acceso_local_local",
        "desc_situacion_local_local",

        # Información de terraza
        "superficie_es",
        "superficie_ra",
        "mesas_es",
        "mesas_ra",
        "sillas_es",
        "sillas_ra",
        "desc_situacion_terraza",
        "desc_periodo_terraza",

        # Horarios ya convertidos previamente
        "hora_cierre_num",

        # Fechas
        "fx_carga_local",
        "fx_carga_terraza",

        # Otros útiles
        "desc_vial_edificio_local",
        "desc_vial_acceso",
        "cod_postal_local",
        "cod_postal_terraza",
    ]

    # Filtrar solo columnas que existen realmente
    columnas_presentes = [c for c in columnas if c in df.columns]

    # -----------------------
    # 2. Crear dataset reducido
    # -----------------------
    df_out = df[columnas_presentes].copy()

    # -----------------------
    # 3. Limpieza básica
    # -----------------------
    df_out = df_out.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

    # Convertir fechas
    for col in ["fx_carga_local", "fx_carga_terraza"]:
        if col in df_out.columns:
            df_out[col] = pd.to_datetime(df_out[col], errors="coerce")

    # Convertir numéricos
    for col in ["superficie_es", "superficie_ra",
                "mesas_es", "mesas_ra", "sillas_es", "sillas_ra",
                "hora_cierre_num"]:
        if col in df_out.columns:
            df_out[col] = pd.to_numeric(df_out[col], errors="coerce")

    # -----------------------
    # 4. Añadir claves normalizadas
    # -----------------------

    # Normalizar nombres administrativos
    if "desc_distrito_local_local" in df_out.columns:
        df_out["distrito"] = df_out["desc_distrito_local_local"].str.strip()

    if "desc_barrio_local_local" in df_out.columns:
        df_out["barrio"] = df_out["desc_barrio_local_local"].str.strip()

    # Crear clave de unión con df_09 (renta)
    if "id_seccion_censal_local" in df_out.columns:
        df_out["seccion_censal"] = df_out["id_seccion_censal_local"]

    # Crear clave de unión con df_05 (ruido + estaciones)
    df_out["latitud"] = pd.to_numeric(df.get("coordenada_y_local_local"), errors="coerce")
    df_out["longitud"] = pd.to_numeric(df.get("coordenada_x_local_local"), errors="coerce")

    # -----------------------
    # 5. Reordenar columnas
    # -----------------------
    orden = [
        "id_local", "rotulo_local",
        "id_terraza", "rotulo_terraza",
        "distrito", "barrio", "cod_barrio_local",
        "seccion_censal",
        "latitud", "longitud",
        "superficie_es", "superficie_ra",
        "mesas_es", "mesas_ra", "sillas_es", "sillas_ra",
        "desc_periodo_terraza", "desc_situacion_terraza",
        "hora_cierre_num",
        "desc_tipo_acceso_local_local", "desc_situacion_local_local",
        "cod_postal_local", "cod_postal_terraza",
        "fx_carga_local", "fx_carga_terraza",
    ]

    columnas_finales = [c for c in orden if c in df_out.columns]
    df_out = df_out[columnas_finales]

    return df_out


In [29]:
#Creamos el dataset optimizado y reducido
df_06_locales_master = crear_df_locales_master(df_locales_terrazas)
df_06_locales_master.head()

,id_local,rotulo_local,id_terraza,rotulo_terraza,distrito,barrio,cod_barrio_local,seccion_censal,latitud,longitud,superficie_es,superficie_ra,mesas_es,mesas_ra,sillas_es,sillas_ra,desc_periodo_terraza,desc_situacion_terraza,hora_cierre_num,desc_tipo_acceso_local_local,desc_situacion_local_local,cod_postal_local,cod_postal_terraza,fx_carga_local,fx_carga_terraza
0,10000003,VITACA,2038.0,VITACA,CENTRO,JUSTICIA,4.0,1088.0,4475338.53,440554.59,6.48,6.48,2.0,2.0,8.0,8.0,Anual,Abierta,NaN,Puerta Calle,Abierto,28004.0,28004.0,2026-03-05,2026-05-03
1,10000004,ZAATAR & CO,19621.0,ZAATAR & CO,CENTRO,UNIVERSIDAD,5.0,1106.0,4475591.53,439945.60,7.02,7.02,3.0,3.0,9.0,9.0,Anual,Abierta,2.0,Puerta Calle,Abierto,28015.0,28015.0,2026-03-05,2026-05-03
2,10000013,HOTEL MEDIODIA,NaN,NaN,CENTRO,EMBAJADORES,2.0,1058.0,4473326.52,441199.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puerta Calle,Abierto,28012.0,NaN,2026-03-05,NaT
3,10000044,V.M. VINOMANIA,NaN,NaN,CENTRO,PALACIO,1.0,1014.0,4473550.53,439722.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puerta Calle,Cerrado,28005.0,NaN,2026-03-05,NaT
4,10000052,MUNE,NaN,NaN,CENTRO,JUSTICIA,4.0,1082.0,4475142.52,440875.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puerta Calle,Abierto,28004.0,NaN,2026-03-05,NaT


In [31]:
df_06_locales_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203241 entries, 0 to 203240
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   id_local                      203241 non-null  int64         
 1   rotulo_local                  149743 non-null  object        
 2   id_terraza                    6467 non-null    float64       
 3   rotulo_terraza                6467 non-null    object        
 4   distrito                      203240 non-null  object        
 5   barrio                        203240 non-null  object        
 6   cod_barrio_local              203240 non-null  float64       
 7   seccion_censal                203240 non-null  float64       
 8   latitud                       203241 non-null  float64       
 9   longitud                      203241 non-null  float64       
 10  superficie_es                 6467 non-null    float64       
 11  superficie_ra

In [37]:
#Instalamos el paquete para convertir coordenadas
!pip install pyproj

   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   -------------- ------------------------- 2.4/6.3 MB 21.6 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 24.8 MB/s  0:00:00


In [39]:
df_06_locales_master.columns.tolist()

['id_local',
 'rotulo_local',
 'id_terraza',
 'rotulo_terraza',
 'distrito',
 'barrio',
 'cod_barrio_local',
 'seccion_censal',
 'latitud',
 'longitud',
 'superficie_es',
 'superficie_ra',
 'mesas_es',
 'mesas_ra',
 'sillas_es',
 'sillas_ra',
 'desc_periodo_terraza',
 'desc_situacion_terraza',
 'hora_cierre_num',
 'desc_tipo_acceso_local_local',
 'desc_situacion_local_local',
 'cod_postal_local',
 'cod_postal_terraza',
 'fx_carga_local',
 'fx_carga_terraza']

In [40]:
df_06_locales_master.head()

,id_local,rotulo_local,id_terraza,rotulo_terraza,distrito,barrio,cod_barrio_local,seccion_censal,latitud,longitud,superficie_es,superficie_ra,mesas_es,mesas_ra,sillas_es,sillas_ra,desc_periodo_terraza,desc_situacion_terraza,hora_cierre_num,desc_tipo_acceso_local_local,desc_situacion_local_local,cod_postal_local,cod_postal_terraza,fx_carga_local,fx_carga_terraza
0,10000003,VITACA,2038.0,VITACA,CENTRO,JUSTICIA,4.0,1088.0,4475338.53,440554.59,6.48,6.48,2.0,2.0,8.0,8.0,Anual,Abierta,NaN,Puerta Calle,Abierto,28004.0,28004.0,2026-03-05,2026-05-03
1,10000004,ZAATAR & CO,19621.0,ZAATAR & CO,CENTRO,UNIVERSIDAD,5.0,1106.0,4475591.53,439945.60,7.02,7.02,3.0,3.0,9.0,9.0,Anual,Abierta,2.0,Puerta Calle,Abierto,28015.0,28015.0,2026-03-05,2026-05-03
2,10000013,HOTEL MEDIODIA,NaN,NaN,CENTRO,EMBAJADORES,2.0,1058.0,4473326.52,441199.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puerta Calle,Abierto,28012.0,NaN,2026-03-05,NaT
3,10000044,V.M. VINOMANIA,NaN,NaN,CENTRO,PALACIO,1.0,1014.0,4473550.53,439722.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puerta Calle,Cerrado,28005.0,NaN,2026-03-05,NaT
4,10000052,MUNE,NaN,NaN,CENTRO,JUSTICIA,4.0,1082.0,4475142.52,440875.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puerta Calle,Abierto,28004.0,NaN,2026-03-05,NaT


In [47]:
#Conversión de coordenadas UTM a EPSG
from pyproj import Transformer

# Transformador UTM (EPSG:25830) → WGS84 (EPSG:4326)
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)

def convertir_utm(df, x_col="longitud", y_col="latitud"):
    """
    Convierte coordenadas UTM ETRS89 / EPSG:25830 (Madrid)
    a coordenadas geográficas WGS84 (longitud, latitud).
    x_col = columna UTM X
    y_col = columna UTM Y
    """
    xs = df[x_col].astype(float).values
    ys = df[y_col].astype(float).values

    lon, lat = transformer.transform(xs, ys)

    df["longitud_wgs84"] = lon
    df["latitud_wgs84"] = lat

    return df

# Aplicar conversión usando tus columnas reales
df_06_locales_master = convertir_utm(df_06_locales_master)


In [ ]:
# EXTRAEMOS LOS DATOS PROCESADOS A LOCAL, PARA LA SIGUIENTE FASE DE ANALISIS

In [48]:
# ----------------------------------------------------------------------
# Guardar los DataFrames procesados localmente
# ----------------------------------------------------------------------
#df_061.to_csv("df_061_censo-locales_processed.csv", index=False)
#df_062.to_csv("df_062_censo-locales_terrazas_processed.csv", index=False)
#df_locales_terrazas.to_csv("df_06_locales_terrazas_processed.csv", index=False)

#fichero reducido con los datos para el análisis
df_06_locales_master.to_csv("df_06_locales_terrazas_processed.csv", index=False)
